10只低波动：跨年度与非银行复算

保留信号、90%目标和现有成本。六个原账户复用、六个非银行账户新增。2021-09-01—2025-08-06，封存价格未使用。

In [1]:
from pathlib import Path
import json,hashlib
import pandas as pd
import numpy as np
root=Path('/Users/guo/Documents/stocks/quant-research')
run=root/'artifacts/lowvol-ten-nonbank-20261013-v1'
a=json.loads((run/'independent-audit.json').read_text())
verified=0
for path,digest in a['source_hashes'].items():
    if '/results/' in path or path.endswith(('protocol.json','implementation-freeze.json')):
        assert hashlib.sha256((root/path).read_bytes()).hexdigest()==digest,path
        verified+=1
for path,digest in json.loads((run/'results/original-account-bindings.json').read_text()).items():
    assert hashlib.sha256((root/path).read_bytes()).hexdigest()==digest,path
print({'audited_accounts':a['accounts'],'sealed_price_rows_read':a['sealed_price_rows_read'],'verified_bindings':verified})
rows=pd.read_csv(run/'results/comparison.csv')
print(rows[['variant','scenario','net_return','max_drawdown','mean_exposure']].to_string(index=False))


{'audited_accounts': 12, 'sealed_price_rows_read': 0, 'verified_bindings': 80}
 variant    scenario  net_return  max_drawdown  mean_exposure
original        base    0.580015      0.112438       0.898932
 nonbank        base    0.475045      0.162970       0.894748
original double_cost    0.443063      0.126673       0.901457
 nonbank double_cost    0.348401      0.178194       0.898031
original      delay1    0.450634      0.127542       0.897160
 nonbank      delay1    0.397514      0.144631       0.893380
original      phase5    0.288863      0.133098       0.894538
 nonbank      phase5    0.138073      0.168668       0.893873
original     phase10    0.139605      0.201281       0.891504
 nonbank     phase10   -0.158697      0.279714       0.889171
original     phase15    0.168474      0.220757       0.881388
 nonbank     phase15   -0.096604      0.341215       0.881079


In [2]:
annual=pd.read_csv(run/'annual-detail.csv')
removed=pd.read_csv(run/'year-removal.csv')
for x in rows.itertuples():
    if x.scenario not in ['base','phase5','phase10','phase15']:continue
    y=annual[(annual.variant==x.variant)&annual.scenario.eq(x.scenario)]
    assert abs(np.prod(1+y.net_return)-1-x.net_return)<1e-10
    for d in removed[(removed.variant==x.variant)&removed.scenario.eq(x.scenario)].itertuples():
        annual_ret=y.loc[y.year.eq(d.removed_year),'net_return'].iloc[0]
        assert abs((1+x.net_return)/(1+annual_ret)-1-d.remaining_observed_return)<1e-10
print(pd.read_csv(run/'annual-summary.csv').to_string(index=False))
print(removed[removed.removed_year.eq(2024)].to_string(index=False))
print('Calendar chains and leave-year-out identities verified.')


 variant  year  full_year  median_net_return  min_net_return  max_net_return  positive_phase_count  max_within_year_drawdown
 nonbank  2021      False           0.063978        0.005278        0.136009                     4                  0.090926
 nonbank  2022       True          -0.143821       -0.236777        0.010468                     1                  0.279257
 nonbank  2023       True          -0.001227       -0.022236        0.031420                     2                  0.135571
 nonbank  2024       True           0.213097        0.128811        0.299733                     4                  0.132467
 nonbank  2025      False          -0.054498       -0.075195       -0.025140                     0                  0.118219
original  2021      False           0.004492       -0.018189        0.061299                     2                  0.071732
original  2022       True          -0.084003       -0.179336        0.075985                     1                  0.220757


In [3]:
print(pd.read_csv(run/'rolling-252.csv').to_string(index=False))
decision=json.loads((run/'decision.json').read_text())
print(json.dumps(decision,ensure_ascii=False,indent=2))
assert decision['continue_direction'] is False
assert decision['sealed_returns_used'] is False


 variant scenario  windows  positive_fraction  min_return  median_return  max_return worst_window_end  overlapping_not_independent
original     base      700           0.991429   -0.013021       0.097653    0.379864       2024-05-23                         True
 nonbank     base      700           0.840000   -0.069424       0.095739    0.354841       2024-02-05                         True
original   phase5      700           0.794286   -0.112316       0.109287    0.274179       2022-09-27                         True
 nonbank   phase5      700           0.588571   -0.097448       0.033454    0.225505       2022-10-13                         True
original  phase10      700           0.784286   -0.165079       0.091357    0.228262       2022-10-24                         True
 nonbank  phase10      700           0.394286   -0.212614      -0.020393    0.146401       2022-10-13                         True
original  phase15      700           0.688571   -0.184871       0.052471    0.35795

年度剔除是已发生收益的集中度诊断，不是可交易策略；滚动窗口与起点均相关。去留判断是公开历史中的研发资源决定，不否定所有低波动投资。当前10只版本淘汰，封存期保留。